In [1]:
"""
Bronze - Autopartage (Citiz + concurrents)

Télécharge les flux GBFS bruts dans le Lakehouse Microsoft Fabric.
Photo unique : chaque exécution remplace les fichiers de l'exécution précédente.

Fonctionnement :
  1. Chaque lien (organisation ou dataset data.gouv / transport.data.gouv)
     est résolu via l'API data.gouv.fr pour retrouver les datasets.
  2. Dans chaque dataset, seules les ressources GBFS sont gardées
     (les exports CSV/JSON qui doublonnent le GBFS sont ignorés).
  3. Chaque point d'entrée GBFS (gbfs.json ou manifest) est lu, puis tous
     les flux qu'il référence sont téléchargés tels quels.
"""

from pathlib import Path
from urllib.parse import urlparse
from collections import Counter
import hashlib
import re

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# ============================================================
# Configuration : un dossier Bronze -> liste de liens
# ============================================================
SOURCES = {
    "gbfs_citiz": [
        "https://www.data.gouv.fr/organizations/reseau-citiz/datasets",
        "https://transport.data.gouv.fr/datasets/vehicules-en-libre-service-autopartage-citiz",
    ],
    "gbfs_concurrent": [
        "https://transport.data.gouv.fr/datasets/gbfs-autopartage-publique-clem",
        "https://transport.data.gouv.fr/datasets/flotte-getaround-en-libre-service-france",
        "https://transport.data.gouv.fr/datasets/remi-autopartage-stations",
        "https://transport.data.gouv.fr/datasets/optymo-auto-en-libre-service",
    ],
}

DATAGOUV_API = "https://www.data.gouv.fr/api/1"
TIMEOUT = 30


# ============================================================
# Vérification du Lakehouse
# ============================================================
fabric_path = Path("/lakehouse/default/Files")

if not fabric_path.is_dir():
    raise RuntimeError(
        "Lakehouse Fabric introuvable (/lakehouse/default/Files). "
        "Vérifie qu'un Lakehouse par défaut est bien attaché au notebook."
    )

BRONZE_DIR = fabric_path / "bronze"
print(f"Utilisation du Lakehouse Microsoft Fabric : {BRONZE_DIR}")


# ============================================================
# Session HTTP (avec quelques relances en cas d'erreur serveur)
# ============================================================
session = requests.Session()
session.headers["User-Agent"] = "hackathon-bronze-autopartage/1.0"
session.mount(
    "https://",
    HTTPAdapter(max_retries=Retry(total=3, backoff_factor=1,
                                  status_forcelist=[429, 500, 502, 503, 504])),
)


def fetch(url):
    resp = session.get(url, timeout=TIMEOUT)
    resp.raise_for_status()
    return resp


def slugify(text):
    return re.sub(r"[^a-z0-9_]+", "-", str(text).lower()).strip("-") or "inconnu"


def normalize(url):
    return url.strip().rstrip("/")


# ============================================================
# Étape 1 : lien -> datasets data.gouv.fr
# ============================================================
def datasets_from_link(link):
    """Accepte un lien d'organisation ou de dataset (data.gouv.fr ou
    transport.data.gouv.fr, qui partagent les mêmes slugs)."""
    parts = [p for p in urlparse(link).path.split("/") if p]

    if "organizations" in parts:
        org = parts[parts.index("organizations") + 1]
        url = f"{DATAGOUV_API}/organizations/{org}/datasets/?page_size=100"
        datasets = []
        while url:
            page = fetch(url).json()
            if isinstance(page, list):          # réponse non paginée
                datasets.extend(page)
                break
            datasets.extend(page.get("data", []))
            url = page.get("next_page")
        return datasets

    if "datasets" in parts:
        slug = parts[parts.index("datasets") + 1]
        return [fetch(f"{DATAGOUV_API}/datasets/{slug}/").json()]

    raise ValueError(f"Lien non reconnu (ni organisation ni dataset) : {link}")


# ============================================================
# Étape 2 : dataset -> points d'entrée GBFS
# ============================================================
def gbfs_entry_points(dataset):
    urls = []
    for res in dataset.get("resources", []):
        fmt = (res.get("format") or "").lower()
        url = res.get("url") or ""
        if fmt == "gbfs" or ("gbfs" in url.lower() and fmt != "csv"):
            urls.append(url)
    return urls


# ============================================================
# Étape 3 : lecture des index GBFS (v1/v2, v3, manifest)
# ============================================================
def is_manifest(doc):
    data = doc.get("data")
    return isinstance(data, dict) and isinstance(data.get("datasets"), list)


def manifest_systems(doc):
    """Manifest GBFS v3 : liste de systèmes, chacun avec ses versions."""
    def version_key(v):
        return tuple(int(x) for x in str(v.get("version", "0")).split(".") if x.isdigit())

    for ds in doc["data"]["datasets"]:
        versions = [v for v in ds.get("versions", []) if v.get("url")]
        if versions:
            best = max(versions, key=version_key)
            yield ds.get("system_id", "systeme"), best["url"]


def extract_feeds(doc):
    """gbfs.json v3 : data.feeds / v1-v2 : data.<langue>.feeds"""
    data = doc.get("data")
    if not isinstance(data, dict) or not data:
        return []
    if isinstance(data.get("feeds"), list):
        return data["feeds"]
    lang = "fr" if "fr" in data else next(iter(data))
    block = data.get(lang)
    return block.get("feeds", []) if isinstance(block, dict) else []


# ============================================================
# Écriture (sans réécrire un fichier identique)
# ============================================================
stats = Counter()
seen_urls = set()


def save(target_dir, filename, content):
    path = target_dir / filename
    if path.exists():
        old = hashlib.sha256(path.read_bytes()).hexdigest()
        if old == hashlib.sha256(content).hexdigest():
            stats["inchangé"] += 1
            print(f"  [=]   {filename} (identique, non réécrit)")
            return
    path.write_bytes(content)
    stats["écrit"] += 1
    print(f"  [OK]  {filename} ({len(content) / 1024:.1f} Ko)")


def process_gbfs(index_url, prefix, target_dir):
    key = normalize(index_url)
    if key in seen_urls:
        stats["doublon"] += 1
        print(f"  [~]   déjà téléchargé dans cette exécution : {index_url}")
        return
    seen_urls.add(key)

    resp = fetch(index_url)
    doc = resp.json()

    # Manifest (ex. Getaround) : un index de plusieurs systèmes
    if is_manifest(doc):
        save(target_dir, f"{prefix}__manifest.json", resp.content)
        for system_id, url in manifest_systems(doc):
            process_gbfs(url, f"{prefix}-{slugify(system_id)}", target_dir)
        return

    # gbfs.json classique : on garde l'index puis on suit chaque flux
    save(target_dir, f"{prefix}__gbfs.json", resp.content)

    for feed in extract_feeds(doc):
        name, url = feed.get("name"), feed.get("url")
        if not url or name == "gbfs":
            continue
        if normalize(url) in seen_urls:
            stats["doublon"] += 1
            continue
        seen_urls.add(normalize(url))
        try:
            r = fetch(url)
            r.json()  # vérifie que c'est bien du JSON, pas une page d'erreur
            save(target_dir, f"{prefix}__{slugify(name)}.json", r.content)
        except (requests.RequestException, ValueError) as e:
            stats["erreur"] += 1
            print(f"  [ERR] flux {name} ({url}) : {e}")


# ============================================================
# Exécution
# ============================================================
for folder, links in SOURCES.items():
    target_dir = BRONZE_DIR / folder
    target_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n=== {folder} ===")

    for link in links:
        try:
            datasets = datasets_from_link(link)
        except (requests.RequestException, ValueError) as e:
            stats["erreur"] += 1
            print(f"[ERR] lien {link} : {e}")
            continue

        for ds in datasets:
            slug = slugify(ds.get("slug", ds.get("id", "dataset")))
            entry_points = gbfs_entry_points(ds)
            if not entry_points:
                stats["sans_gbfs"] += 1
                print(f"[--] {slug} : aucune ressource GBFS, ignoré")
                continue

            print(f"[{slug}]")
            for i, url in enumerate(entry_points, start=1):
                prefix = slug if len(entry_points) == 1 else f"{slug}-{i}"
                try:
                    process_gbfs(url, prefix, target_dir)
                except (requests.RequestException, ValueError) as e:
                    stats["erreur"] += 1
                    print(f"  [ERR] index {url} : {e}")

print("\n=== Récapitulatif ===")
print(f"Fichiers écrits            : {stats['écrit']}")
print(f"Fichiers inchangés         : {stats['inchangé']}")
print(f"Doublons évités            : {stats['doublon']}")
print(f"Datasets sans GBFS ignorés : {stats['sans_gbfs']}")
print(f"Erreurs                    : {stats['erreur']}")

In [1]:
import requests

base = "/lakehouse/default/Files/bronze/delinquance_ssmsi"
files = {
    "delinquance_communale.parquet": "https://www.data.gouv.fr/api/1/datasets/r/604d71b8-337d-4869-9226-49e01bae87df",
    "delinquance_departementale.csv": "https://www.data.gouv.fr/api/1/datasets/r/2b27a675-e3bf-41ef-a852-5fb9ab483967",
    "delinquance_regionale.csv": "https://www.data.gouv.fr/api/1/datasets/r/75753852-aca0-4c53-bfd4-a29513358d2f",
}
import os; os.makedirs(base, exist_ok=True)
for name, url in files.items():
    r = requests.get(url, allow_redirects=True, timeout=120)
    r.raise_for_status()
    with open(f"{base}/{name}", "wb") as f:
        f.write(r.content)